# DES Y3 joint shear: evaluate the Cobaya model and its Fisher derivatives

This notebook follows `EXAMPLE_EVALUATE1.yaml`, just as the LSST Y1 evaluate
notebooks follow their numbered examples. Cobaya owns CAMB, the supplied
bispectrum theory block and `likelihood/joint_shear.py`. The compiled interface
owns numerical settings and the selected data/covariance. There is no separate
preparation stage, `PhysicalProvider`, or alternative prediction pipeline.

The Fisher section calls the shared CosmoLike functions directly. It uses a
Gaussian, fixed-covariance approximation; it does not reproduce the finite-mock
posterior merely because it uses the same measured rows. Its parameter list and
priors must be stated before interpreting marginalized errors.

In [ ]:
%matplotlib inline
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"

from pathlib import Path
import sys
import numpy as np
from matplotlib import pyplot as plt
from cobaya.yaml import yaml_load_file
from cobaya.model import get_model

project = Path.cwd().resolve()
while not (project / "EXAMPLE_EVALUATE1.yaml").exists():
    if project == project.parent:
        raise FileNotFoundError("Open this notebook from the DES project.")
    project = project.parent
cocoa = project.parents[1]
sys.path.insert(0, str(project / "interface"))
sys.path.insert(0, str(cocoa / "external_modules/code/cosmolike_core"))
import cosmolike_des_3pcf_interface as ci
import cosmolike_notebook_utils.fisher as cnu

# The ordinary Cocoa YAML uses paths relative to the activated Cocoa folder.
os.chdir(cocoa)
info = yaml_load_file(str(project / "EXAMPLE_EVALUATE1.yaml"))
fiducial = dict(info["sampler"]["evaluate"]["override"])
info.pop("sampler")
info.pop("output", None)
info["likelihood"]["des_3pcf.joint_shear"]["omp_threads"] = 1


## Evaluate exactly the model described by the YAML

Build one model at a time: the compiled CosmoLike dataset is process-global.
`loglike` evaluates the theory blocks before the likelihood. The vector retained
by the likelihood is therefore the one it actually scored, including fixed
nuisance values. The native selector applies the same mask and optional
compression as the likelihood; the notebook does not duplicate that logic.

In [ ]:
model = get_model(info)
likelihood = model.likelihood["des_3pcf.joint_shear"]
loglike = model.loglike(fiducial, return_derived=False, cached=False)
if not np.isfinite(loglike):
    raise RuntimeError("The fiducial likelihood evaluation failed.")
mean = ci.select_joint_data_vector(prediction=likelihood.last_datavector)
data = ci.get_joint_data()
covariance = ci.get_joint_covariance()
print("Selected rows:", mean.size)
print("Log likelihood:", loglike)


## Inspect residuals before differentiating

Divide each residual by its own standard deviation to show scale-dependent
mismatches. These plotted values remain correlated; their squared sum is not
the joint chi-squared. The actual likelihood uses the entire covariance.

In [ ]:
sigma = np.sqrt(np.diag(covariance))
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(np.arange(mean.size), (mean-data)/sigma)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xlabel("Selected data-vector row")
ax.set_ylabel("(Theory − data) / standard deviation")
plt.show()


## Fisher information through the shared machinery

For a parameter displacement, run the same Cobaya model again. No separate
CAMB object is prepared here. `cached=False` forces a real evaluation. Flat
prior edges do not truncate the derivative because `loglike` omits the prior.

The derivative matrix has one row per selected observable and one column per
varied parameter. The shared routine evaluates fourth-order centered stencils
at both an absolute step $h$ and $h/2$. Agreement is a necessary diagnostic,
not proof of integration convergence.

The three-parameter example below holds all other parameters fixed. Its inverse
is a **conditional** forecast, not a full DES nuisance-marginalized result.
Expand the parameter list, supply justified Gaussian prior widths and repeat
accuracy/step checks before claiming a survey forecast.

In [ ]:
def get_dv(parameters):
    # Evaluate the same YAML theory and likelihood for every stencil point.
    value = model.loglike(parameters, return_derived=False, cached=False)
    if not np.isfinite(value):
        raise RuntimeError("A displaced likelihood evaluation failed.")
    return ci.select_joint_data_vector(prediction=likelihood.last_datavector)

steps = {"omega_m": 0.003, "sigma8": 0.005, "a1": 0.01}
# No Gaussian information is inferred from uniform prior bounds.
priors = {}
names, center, derivatives_h, derivatives_h2 = cnu.differentiate(
    predict=get_dv,
    parameters=fiducial,
    steps=steps,
)
fisher_h = cnu.fisher_matrix(
    derivatives=derivatives_h,
    names=names,
    covariance=covariance,
    priors=priors,
)
fisher_h2 = cnu.fisher_matrix(
    derivatives=derivatives_h2,
    names=names,
    covariance=covariance,
    priors=priors,
)
errors_h = np.sqrt(np.diag(cnu.parameter_covariance(fisher_h, names)))
errors_h2 = np.sqrt(np.diag(cnu.parameter_covariance(fisher_h2, names)))
for name, first, second in zip(names, errors_h, errors_h2):
    print(name, "conditional errors at h, h/2:", first, second)


## Retain the inputs needed to interpret a result

Save the derivatives as well as the Fisher matrices: they are the expensive
part and allow later inspection. Outputs belong under `chains/`. Keep the YAML,
parameter list, step sizes, accuracy settings and prior choices with a published
result. Changing the covariance, parameter set or model changes the forecast.

In [ ]:
output = project / "chains/EXAMPLE_EVALUATE1_fisher.npz"
output.parent.mkdir(parents=True, exist_ok=True)
np.savez(
    output,
    parameter_names=np.asarray(names),
    steps=np.asarray(list(steps.values())),
    fiducial_mean=center,
    covariance=covariance,
    derivatives_h=derivatives_h,
    derivatives_h2=derivatives_h2,
    fisher_h=fisher_h,
    fisher_h2=fisher_h2,
)
model.close()
